In [1]:
"""Notebook Lens sample cell: stream text, embed it, and run a small sanity check."""

from __future__ import annotations

import os
import warnings
from collections import Counter, defaultdict
from itertools import islice


warnings.filterwarnings("ignore", message="IProgress not found.*")

DATASET_NAME = os.environ.get("HF_EMBED_DATASET", "fancyzhx/ag_news")
DATASET_CONFIG = os.environ.get("HF_EMBED_CONFIG") or None
DATASET_SPLIT = os.environ.get("HF_EMBED_SPLIT", "train")
TEXT_FIELD = os.environ.get("HF_EMBED_TEXT_FIELD", "text")
LABEL_FIELD = os.environ.get("HF_EMBED_LABEL_FIELD", "label")
ROW_LIMIT = int(os.environ.get("HF_EMBED_ROWS", "400"))
MODEL_NAME = os.environ.get("HF_EMBED_MODEL", "sentence-transformers/all-MiniLM-L6-v2")


def load_dependencies():
    missing = []
    try:
        from datasets import load_dataset
    except ModuleNotFoundError:
        load_dataset = None
        missing.append("datasets")
    try:
        from sentence_transformers import SentenceTransformer
    except ModuleNotFoundError:
        SentenceTransformer = None
        missing.append("sentence-transformers")
    if missing:
        print(f"Missing optional dependencies: {', '.join(missing)}")
        print("Install with: python -m pip install datasets sentence-transformers")
        return None, None
    return load_dataset, SentenceTransformer


def average(vectors: list[list[float]]) -> list[float]:
    return [sum(values) / len(vectors) for values in zip(*vectors, strict=False)]


def dot(left: list[float], right: list[float]) -> float:
    return sum(a * b for a, b in zip(left, right, strict=False))


load_dataset, sentence_transformer_cls = load_dependencies()
if load_dataset is None or sentence_transformer_cls is None:
    HF_SAMPLE_STATUS = "missing optional dependency"
else:
    kwargs = {"split": DATASET_SPLIT, "streaming": True}
    if DATASET_CONFIG:
        stream = load_dataset(DATASET_NAME, DATASET_CONFIG, **kwargs)
    else:
        stream = load_dataset(DATASET_NAME, **kwargs)

    texts: list[str] = []
    labels: list[str] = []
    for row in islice(stream, ROW_LIMIT):
        text = str(row.get(TEXT_FIELD) or "").strip()
        if not text:
            continue
        texts.append(text[:512])
        labels.append(str(row.get(LABEL_FIELD, "unknown")))

    print(f"dataset: {DATASET_NAME}")
    print(f"split: {DATASET_SPLIT}")
    print(f"rows_embedded: {len(texts)}")
    print(f"model: {MODEL_NAME}")

    model = sentence_transformer_cls(MODEL_NAME)
    encoded = model.encode(
        texts,
        batch_size=32,
        normalize_embeddings=True,
        show_progress_bar=True,
    )
    vectors = [list(map(float, vector)) for vector in encoded]

    by_label: dict[str, list[list[float]]] = defaultdict(list)
    for label, vector in zip(labels, vectors, strict=False):
        by_label[label].append(vector)
    centroids = {label: average(group) for label, group in by_label.items()}

    correct = 0
    for label, vector in zip(labels, vectors, strict=False):
        predicted = max(centroids.items(), key=lambda item: dot(vector, item[1]))[0]
        correct += int(predicted == label)

    counts = Counter(labels)
    print("label_counts:")
    for label, count in counts.most_common():
        print(f"- {label}: {count}")
    accuracy = correct / len(labels) if labels else 0.0
    print(f"nearest_centroid_self_check_accuracy: {accuracy:.3f}")
    HF_SAMPLE_STATUS = "ok"


dataset: fancyzhx/ag_news
split: train
rows_embedded: 60
model: sentence-transformers/all-MiniLM-L6-v2


Batches: 100%|██████████| 2/2 [00:00<00:00,  8.41it/s]

label_counts:
- 2: 60
nearest_centroid_self_check_accuracy: 1.000
